In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import auc

sys.path.append(os.path.abspath("../"))

from util import dataset as u_dataset
from util import metrics as u_metrics

In [ ]:
log_path = Path("../../logs/fit/")

In [ ]:
# Okabe-Ito color palette
okabe_ito_colors = [
    "#E69F00",  # Orange
    "#56B4E9",  # Sky Blue
    "#009E73",  # Bluish Green
    "#F0E442",  # Yellow
    "#0072B2",  # Blue
    "#D55E00",  # Vermillion
    "#CC79A7",  # Reddish Purple
    "#000000",  # Black
]


def plot_pr_curve(mode: str, object_name: str, specification_strings: list[str]):
    base_dir = Path(log_path, mode)
    folders = [f for f in base_dir.iterdir() if f.is_dir()]
    if len(folders) != 1:
        raise ValueError(f"Expected exactly one folder in {base_dir}, found {len(folders)}")
    path_to_model = folders[0]

    # Load data for all distances
    all_recalls = []
    all_precisions = []
    candidate_strings = []
    for spec in specification_strings:
        recalls = np.load(
            path_to_model.as_posix() + f"/evaluation/{spec}/recalls/{object_name}.npy"
        )
        precisions = np.load(
            path_to_model.as_posix() + f"/evaluation/{spec}/precisions/{object_name}.npy"
        )
        processed_pr = u_metrics.process_precision_recall(precisions, recalls)
        all_recalls.append(processed_pr["recalls"])
        all_precisions.append(processed_pr["precisions"])

        n_candidates = spec.split("_")[-1].split("-")
        if object_name == u_dataset.CategoryNames.BALL.value:
            candidate_strings.append(n_candidates[0])
        elif object_name == u_dataset.CategoryNames.PENALTYMARK.value:
            candidate_strings.append(n_candidates[1])
        elif u_dataset.CategoryNames.INTERSECTIONS.value in object_name:
            candidate_strings.append(n_candidates[2])

    plot(all_recalls, all_precisions, candidate_strings, object_name)


def plot(all_recalls, all_precisions, candidate_strings, object_name):
    fig, ax = plt.subplots(figsize=(5, 4))

    for i, (recalls, precisions, spec) in enumerate(
        zip(all_recalls, all_precisions, candidate_strings, strict=False)
    ):
        title = f"Precision-Recall Curve , object: {object_name} "  # + r"$K_c$:" + f"{candidate_string}"

        distance_string = spec.split("-")[0].split("_")[-1]
        candidates_string = spec.split("_")[-1].split("-")[-1]

        recalls = np.append(recalls, max(recalls))
        precisions = np.append(precisions, 0)
        pr_auc = auc(recalls, precisions)
        color = okabe_ito_colors[i % len(okabe_ito_colors)]
        ax.plot(
            recalls,
            precisions,
            color=color,
            linewidth=2,
            # label=f"{distance_string} m (AP = {pr_auc:.3f})",
            label=r"$K_c$=" + f"{spec} (AP = {pr_auc:.3f})",
        )
        # Optional: Fill under the curve
        # ax.fill_between(recalls, precisions, alpha=0.1, color=color)

    ax.set_xlabel("Recall", fontsize=13)
    ax.set_ylabel("Precision", fontsize=13)
    # ax.set_title(title, fontsize=15)
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.0])
    ax.legend(loc="lower left", fontsize=11)
    ax.grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()
    save_path = "../../plots/n_candidates_pr/"
    os.makedirs(save_path, exist_ok=True)
    plt.savefig(f"{save_path}{object_name}.pdf")
    plt.show()

In [ ]:
def plot_final_curves(mode, distances, n_candidates, categories, intersection_types=None):
    specification_strings = []
    for distance in distances:
        for n_candidate in n_candidates:
            specification_strings.append(f"d_{distance}-K_{n_candidate}")

    for object in categories:
        if object == u_dataset.CategoryNames.INTERSECTIONS:
            for type in intersection_types:
                object_name = f"{object.value}_{type.name}"
                plot_pr_curve(mode, object_name, specification_strings)

        else:
            plot_pr_curve(mode, object.value, specification_strings)


### Demonstration of AP interpolation


In [ ]:
# np.random.seed(42)

# recalls = np.linspace(0.0, 1, 100)

# precisions = np.zeros(100)

# for i, r in enumerate(recalls):
#     if r < 0.15:
#         base = 0.50
#         noise = np.random.uniform(-0.04, 0.04)
#     elif r < 0.75:
#         base = 0.50 + (r - 0.15) * (0.47 / 0.60)  # linear rise from 0.50 to ~0.97
#         noise = np.random.uniform(-0.04, 0.04)
#     elif r < 0.85:
#         # small valley: dips down then recovers
#         mid = 0.80
#         depth = 0.12
#         base = 1.0 - depth * np.exp(-((r - mid) ** 2) / (2 * 0.02))
#         noise = np.random.uniform(-0.02, 0.02)
#     else:
#         base = 1.0 - (r - 0.85) * (0.70 / 0.15)  # sharp collapse
#         noise = np.random.uniform(-0.02, 0.02)

#     precisions[i] = np.clip(base + noise, 0.0, 1.0)

# print("recalls =", recalls.tolist())
# print("precisions =", precisions.tolist())

# precisions_interp = np.maximum.accumulate(precisions[::-1])[::-1]

# plot([recalls], [precisions], ["a"], "")
# plot([recalls], [precisions_interp], ["a"], "")

In [ ]:
categories = [
    u_dataset.CategoryNames.BALL,
    # u_dataset.CategoryNames.PENALTYMARK,
    # u_dataset.CategoryNames.INTERSECTIONS,
]

timestamp = "20260711-000000"
mode = "final"
n_candidates = [
    # "1-1-1",
    # "4-4-11"
    "2-2-2",
    "3-3-3",
    "4-4-4",
    "5-5-5",
    # "6-6-6",
    # "7-7-7",
    # "8-8-8",
    # "9-9-9",
    # "10-10-10",
    # "11-11-11",
    # "12-12-12",
    # "13-13-13",
    # "14-14-14",
]
distances = ["9"]
plot_final_curves(mode, distances, n_candidates, categories)

In [ ]:
categories = [
    # u_dataset.CategoryNames.BALL,
    u_dataset.CategoryNames.PENALTYMARK,
    # u_dataset.CategoryNames.INTERSECTIONS,
]


timestamp = "20260711-000000"
mode = "final"
# distances = ["3", "5", "7", "9"]
n_candidates = [
    "1-1-1",
    "2-2-2",
    "3-3-3",
    "4-4-4",
    "5-5-5",
    "6-6-6",
    # "7-7-7",
    # "8-8-8",
    # "9-9-9",
    # "10-10-10",
    # "11-11-11",
    # "12-12-12",
    # "13-13-13",
    # "14-14-14",
]
distances = ["9"]
plot_final_curves(mode, distances, n_candidates, categories)

In [ ]:
categories = [
    # u_dataset.CategoryNames.BALL,
    # u_dataset.CategoryNames.PENALTYMARK,
    u_dataset.CategoryNames.INTERSECTIONS,
]
intersection_types = [
    u_dataset.IntersectionType.L,
    u_dataset.IntersectionType.T,
    u_dataset.IntersectionType.X,
]


timestamp = "20260711-000000"
mode = "final"
n_candidates = [
    # "1-1-1",
    # "2-2-2",
    # "3-3-3",
    # "4-4-4",
    # "5-5-5",
    # "6-6-6",
    "7-7-7",
    "8-8-8",
    "9-9-9",
    "10-10-10",
    "11-11-11",
    "12-12-12",
    "13-13-13",
    # "14-14-14",
]
distances = ["9"]
plot_final_curves(mode, distances, n_candidates, categories, intersection_types)